# Context-Aware Text Generation with LangChain Prompt Templates

Perform Text Generation using a prompt that includes Context

In this notebook, you will learn how to generate an email response to a customer who was not happy with the quality of customer service they received from the customer support engineer. You will provide additional context to the model by including the contents of the actual email received from the unhappy customer.

You will add more complexity with the help of PromptTemplates to leverage the LangChain framework for a similar use case. PromptTemplates allow you to create generic shells which can be populated with information later and obtain model outputs based on different scenarios.

[LangChain](https://python.langchain.com/docs/get_started/introduction.html) is a framework for developing applications powered by language models. The key aspects of this framework allow us to augment the Large Language Models by chaining together various components to create advanced use cases.

Due to the additional context in the prompt, the content produced in this notebook is of much better quality and relevance than the content produced earlier through zero-shot prompts. The prompt used in this notebook creates a custom LangChain prompt template for adding context to the text generation request.

### Scenario

You are Bob, a Customer Service Manager at AnyCompany, and some of your customers are not happy with the customer service and are providing negative feedback on the service provided by customer support engineers. Now, you would like to respond to those customers humbly, apologizing for the poor service and regain their trust. You need the help of a Large Language Model (LLM) to generate a bulk of emails for you, which are human-friendly and personalized to the customer's sentiment from previous email correspondence.

In this scenario, you can leverage the power of LangChain's PromptTemplates to create a generic shell for generating personalized email responses based on the customer's previous email. The PromptTemplate will incorporate the customer's original email content, allowing the LLM to understand the context and sentiment, and then generate a relevant and customized response.

In [0]:
customer_feedback = """
    Hello Bob,  I am very disappointed with the recent experience I had when I called your customer support.
    I was expecting an immediate call back but it took three days for us to get a call back.
    The first suggestion to fix the problem was incorrect. Ultimately the problem was fixed after three days.
    We are very unhappy with the response provided and may consider taking our business elsewhere.
    """

feedback_context = {
    "customerServiceManager": "Bob Smith", 
    "customerName": "John Doe", 
     "feedbackFromCustomer": customer_feedback
}

### Task 1: Environment setup

Configure the development environment and install the required dependencies.

In [0]:
import json
import os
import sys
import warnings

warnings.filterwarnings('ignore')
module_path = ".."
sys.path.append(os.path.abspath(module_path))

The `boto3.client('bedrock-runtime')` method creates a client for interacting with Amazon Bedrock's model inference APIs. It accepts the following:

- `service_name`: The AWS service to connect to, in this case, `bedrock-runtime`.
- `region_name`: The AWS Region where the service is available. It defaults to the value of the `AWS_DEFAULT_REGION` environment variable, or `None` if the variable is not set, allowing Boto3 to resolve the region from its default configuration.

Refer to the [Boto3 documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/core/session.html?utm_source=chatgpt.com#boto3.session.Session.client) for more information.

In [0]:
import boto3

bedrock_client = boto3.client(
    'bedrock-runtime',
    region_name=os.environ.get("AWS_DEFAULT_REGION", None)
)

## Task 2: Invoke the Bedrock LLM Model

In this task, you create an instance of the Bedrock class from llms. This expects a `model_id` which is the Amazon Resource Name (ARN) of the model available in Amazon Bedrock.

Optionally, you can pass a previously created boto3 client as well as some `model_kwargs` which can hold parameters such as `temperature`, `top_p`, `max_token_count`, or `stop_sequences` (more information on parameters can be explored in the Amazon Bedrock console).

Refer to [documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/model-ids-arns.html) for Available text generation model Ids under Amazon Bedrock.

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Note:** The different models support different `model_kwargs`.

In [0]:
from langchain_aws import ChatBedrock
from langchain_core.output_parsers import StrOutputParser

model_id = "meta.llama3-8b-instruct-v1:0"
model_kwargs =  { 
        "max_gen_len": 512,
        "temperature": 0,
        "top_p": 1,
}

chat_model = ChatBedrock(
    client=bedrock_client,
    model_id=model_id,
    model_kwargs=model_kwargs,
)

## Task 3: Create a LangChain custom prompt template

In this task, you will create a template for the prompt that you can pass different input variables on every run. This is useful when you have to generate content with different input variables that you may be fetching from a database.

In the previous task, we hardcoded the prompt. It might be the case that you have multiple customers sending similar negative feedback, and you now want to use each of those customers' emails and respond to them with an apology, but you also want to keep the response a bit personalized. In the following cell, you will explore how you can create a `PromptTemplate` to achieve this pattern.

In [0]:
from langchain_core.prompts import PromptTemplate

multi_var_prompt = PromptTemplate(
    input_variables=["customerServiceManager", "customerName", "feedbackFromCustomer"], 
    template="""
        Create an apology email from the Service Manager {customerServiceManager} at AnyCompany to {customerName} in 
        response to the following feedback that was received from the customer: 
    
        <customer_feedback>
        {feedbackFromCustomer}
        </customer_feedback>
    """
)

prompt = multi_var_prompt.format(**feedback_context)

The function `get_num_tokens()` counts the number of tokens in a text string using the tokenizer associated with the chat model. This helps estimate prompt size and check whether it fits within the model's context window.

In [0]:
num_tokens = chat_model.get_num_tokens(prompt)
print(f"The prompt has {num_tokens} tokens.")

The prompt has 188 tokens.


The function `invoke()` sends the prompt to the chat model and returns the model's response as an AI message object.

In [0]:
direct_response = chat_model.invoke(prompt)
print(direct_response.content)

Here is an apology email from Service Manager Bob Smith at AnyCompany to John Doe:

Subject: Apology and Follow-up on Recent Customer Support Experience

Dear John,

I am writing to express my sincerest apologies for the disappointing experience you had with our customer support team. I am Bob Smith, the Service Manager at AnyCompany, and I take full responsibility for the issues you faced.

I was deeply troubled to learn that it took three days for us to get back to you after your initial call. This is far from the level of service we strive to provide, and I apologize for the frustration and inconvenience this caused.

I also regret that the initial solution we provided was not effective in resolving your issue. I understand that this added to your frustration and disappointment. I want to assure you that we are taking immediate action to address the root cause of the problem and prevent similar situations from occurring in the future.

I want to thank you for bringing this to our at

**`invoke()` in Chat Models vs. Chains**

- `chat_model.invoke(prompt)` sends a prompt directly to the language model and returns the generated response as an AI message object.
- `chain.invoke(input)` executes the entire chain, which may include prompt templates, language models, and output parsers. It receives the input expected by the chain, typically a dictionary.

**LCEL (LangChain Expression Language) does not inherently improve the quality of LLM responses.** When using the same prompt, model, and configuration, both direct invocation and LCEL are expected to produce equivalent results. The main advantage of LCEL is its ability to compose, reuse, and manage more complex processing pipelines.

| Aspect | Direct Invocation | LCEL |
|---|---|---|
| Expected response quality | Equivalent | Equivalent |
| Pipeline organization | Manual | Declarative |
| Component composition | More manual | Simplified with `\|` |
| Pipeline reusability | Requires more manual code | Easier |
| Streaming, async, and batch | Supported by relevant APIs | Standardized through `Runnable` interfaces |

**Conclusion:** Direct invocation is sufficient for simple model calls. LCEL becomes more valuable as pipelines grow in complexity, incorporating prompt templates, output parsers, retrieval (RAG), and multiple processing steps.

**Note:** The argument passed to `chain.invoke()` must match the chain's expected input format. Passing a model response is only appropriate if the chain is designed to accept it.

In [0]:
chain =  multi_var_prompt | chat_model | StrOutputParser()
chain_output = chain.invoke(feedback_context)

print(chain_output)

Here is an apology email from Service Manager Bob Smith at AnyCompany to John Doe:

Subject: Apology and Follow-up on Recent Customer Support Experience

Dear John,

I am writing to express my sincerest apologies for the disappointing experience you had with our customer support team. I am Bob Smith, the Service Manager at AnyCompany, and I take full responsibility for the issues you faced.

I was deeply troubled to learn that it took three days for us to get back to you after your initial call. This is far from the level of service we strive to provide, and I apologize for the frustration and inconvenience this caused.

I also want to apologize for the incorrect solution provided to you. I understand that this added to your frustration and wasted valuable time. I am glad to hear that the problem was ultimately resolved, but I am truly sorry that it took longer than expected.

At AnyCompany, we value our customers and their feedback. Your experience has highlighted an area where we can

You have successfully learned that invoking the LLM without any context might not yield the desired results. By adding context and further using the prompt template to constrain the output from the LLM, you were able to successfully obtain your desired output.

### Try it yourself
- Change the prompts to your specific usecase and evaluate the output of different models.
- Play with the token length to understand the latency and responsiveness of the service.
- Apply different prompt engineering principles to get better outputs.

I hope you enjoyed it!

Visit my [GitHub profile](https://github.com/rafaself).